# Explaining and auditing models

**Lecture 10** · Lecture notes

Applications of Machine Learning — BSc Mathematics of Artificial Intelligence

---

**How to use this notebook.** Read before you run. Every code cell is preceded
by the specification that would produce it — input, output, constraint, check.

A bank pre-approves applicants instantly when a model says their income is
above a band; everyone else goes to a slower manual check. This notebook asks
two questions of that model. **What does it rely on?** — globally, with
permutation importance and partial dependence, and for one applicant, with
Shapley values computed exactly. **For whom does it fail?** — with metrics per
slice of the applicants, three fairness criteria, the proxies that carry a
column the model was never given, and what moving a threshold for one group
does.

The data are the 1994 US census extract known as *Adult*: 48,842 people, and
whether each earned more than 50,000 dollars a year. The bank is a frame; the
numbers are this notebook's own.

**About the specification boxes.** Every code cell in this notebook is preceded
by a box naming four things: the input, the output, the constraint the method
must respect, and a check whose answer you can work out before running
anything. Read the box, answer the check in your head, then run the cell.

Each box then shows two ways of asking an assistant for the cell: the **novice
prompt**, which is what most people would type, and the **pro prompt**, which
names every decision the cell depends on — followed by **why the pro prompt
works better**. That last field is tested, not guessed: each novice prompt was
given to a current AI assistant together with this notebook up to that cell,
and the why reports what came back. Often the assistant gets much of the cell
right unasked; the why then says what it still got wrong, or did without being
required to, and which of the pro's choices settles it. The two prompts are
written for this cell, not recorded: nobody claims either produced the code
below. Neither states the check's answer, because the check is how *you* find
out whether the answer is wrong.


## 1 · Setup

> **Specification · setup**
>
> **input** · nothing
>
> **output** · the libraries, their versions, and one seed for every random choice below
>
> **constraint** · one seed, `RANDOM_STATE = 42`, passed explicitly to every split, model and shuffle — an explanation that changes when the notebook is re-run is not an explanation of the model
>
> **check** · the versions print; scikit-learn must be 1.4 or later, because the forest below is fitted on rows with missing values
>
> ---
>
> **novice prompt** · “import what I need for model explainability”
>
> **pro prompt** · “Import numpy, pandas, scikit-learn and matplotlib, plus `itertools` and `math` for the Shapley enumeration; from scikit-learn import `fetch_openml`, `train_test_split`, `HistGradientBoostingClassifier`, `RandomForestClassifier`, `permutation_importance` and `roc_auc_score`. Print the Python, numpy, pandas and scikit-learn versions. Define `RANDOM_STATE = 42` and use no global seed: every function that draws takes it explicitly. Do not install or import `shap`.”
>
> **why the pro prompt works better** · 'Import what I need for model explainability' is read as a request for the explanation libraries. Asked this way, an assistant typically opens with `pip install shap` or `lime`, which this notebook does not use and which moves the installed versions under every other cell, and it seeds `np.random` globally, so a cell run twice draws differently from the same cell run once. The pro's explicit import list, 'do not install or import `shap`' and one `RANDOM_STATE` passed to every draw make the explanations below a property of the model rather than of the session.
>
> ---
>
> **try** · change `RANDOM_STATE` to 0 and run everything. The split, the model and every number move a little; a conclusion that survives the change is about the data, and one that does not was about the seed.

In [ ]:
# --- setup -------------------------------------------------------------------
import sys, itertools, math, warnings
import numpy as np, pandas as pd, sklearn
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

print(f"python       {sys.version.split()[0]}")
print(f"numpy        {np.__version__}")
print(f"pandas       {pd.__version__}")
print(f"scikit-learn {sklearn.__version__}")

# One seed, passed by hand to every split, model and shuffle below. Nothing
# here relies on a global random state.
RANDOM_STATE = 42
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

## 2 · The data: twelve answers, one target, one attribute kept aside

Each row is a person. The target is whether they earned more than 50,000
dollars. Two columns are **not** given to the model: `sex` and `race`. The
bank's product team removed them, on the reasoning that a model which never
sees an attribute cannot use it — a reasoning section 12 tests. They are kept
beside the rows, for the audit only.

The categorical answers are turned into integer codes, one per level, and a
missing answer stays missing (`NaN`): both models below handle a missing value
natively, and inventing an "unknown" level would be a choice nobody asked for.
The split is stratified on the target *and* on sex, so that every group the
audit reads has the same share in the fit, validation and test rows.

> **Specification · the data**
>
> **input** · the `adult` dataset, version 2, from OpenML
>
> **output** · a float matrix of the twelve features the model sees, the target, the sensitive attributes kept aside, and a fit / validation / test split of 60 / 20 / 20
>
> **constraint** · `sex` and `race` must not be among the model's columns, and must stay aligned with every row; categorical answers become integer codes with a missing answer kept as NaN; the split is stratified on target and sex together
>
> **check** · 48,842 rows; 15 columns arrive and 12 remain as features, 6 of them categorical; the three parts are 29,304, 9,769 and 9,769 rows and add up to the whole
>
> ---
>
> **novice prompt** · “load the adult income dataset and split it”
>
> **pro prompt** · “Fetch `adult` version 2 with `fetch_openml(..., as_frame=True, data_home="datasets")` and assert its shape is (48842, 15). Make `y_all` the indicator of `class == ">50K"` and `female_all` the indicator of `sex == "Female"`; keep `race` as strings. `FEATURES` is every column except class, sex and race; `CATEGORICAL` those of categorical dtype, `CAT_MASK` the boolean mask, `LEVELS` the category names. Replace each categorical column by its codes, with -1 turned into NaN, and convert the whole frame to one float matrix `X_all`. Split row indices 80/20 into rest/test and the rest 75/25 into fit/validation, both stratified on `2 * y_all + female_all` with `RANDOM_STATE`. Print the sizes and the share above 50K overall, for men and for women.”
>
> **why the pro prompt works better** · 'Load the adult income dataset and split it' leaves the sensitive columns where they are. Asked this way, an assistant typically one-hot encodes every categorical column, including `sex` and `race`, so the model is given the attribute the bank promised not to use, and twelve answers become about a hundred columns that no importance or Shapley value below can be read against. It fills the missing answers with the most frequent level and stratifies, if at all, on the target alone. The pro's `FEATURES` without class, sex and race, codes with NaN for a missing answer, and stratification on `2 * y_all + female_all` keep one column per answer and the groups the audit reads equally represented in every part.
>
> ---
>
> **try** · stratify on `y_all` alone. The three parts keep the same share above 50K, while the share of women in the test rows moves by a few tenths of a point — harmless here, and the reason an audit by group stratifies on the group.

In [ ]:
adult = fetch_openml("adult", version=2, as_frame=True, data_home="datasets").frame
assert adult.shape == (48842, 15)

y_all      = (adult["class"] == ">50K").to_numpy().astype(int)
female_all = (adult["sex"] == "Female").to_numpy().astype(int)
race_all   = adult["race"].astype(str).to_numpy()

# The model never sees sex or race: they are kept beside the rows, for the audit.
FEATURES    = [c for c in adult.columns if c not in ("class", "sex", "race")]
CATEGORICAL = [c for c in FEATURES if isinstance(adult[c].dtype, pd.CategoricalDtype)]
CAT_MASK    = np.array([c in CATEGORICAL for c in FEATURES])
LEVELS      = {c: list(adult[c].cat.categories) for c in CATEGORICAL}

coded = adult[FEATURES].copy()
for c in CATEGORICAL:
    # one integer per level; a missing answer stays missing rather than
    # becoming an invented level
    coded[c] = coded[c].cat.codes.astype(float).replace(-1.0, np.nan)
X_all = coded.to_numpy(dtype=float)

strata = 2 * y_all + female_all
rest, i_test = train_test_split(np.arange(len(adult)), test_size=0.2,
                                stratify=strata, random_state=RANDOM_STATE)
i_fit, i_val = train_test_split(rest, test_size=0.25, stratify=strata[rest],
                                random_state=RANDOM_STATE)

X_fit,  y_fit,  female_fit  = X_all[i_fit],  y_all[i_fit],  female_all[i_fit]
X_val,  y_val,  female_val  = X_all[i_val],  y_all[i_val],  female_all[i_val]
X_test, y_test, female_test = X_all[i_test], y_all[i_test], female_all[i_test]
race_test = race_all[i_test]

assert len(FEATURES) == 12 and len(CATEGORICAL) == 6
assert len(i_fit) + len(i_val) + len(i_test) == len(adult)
print(f"rows {len(adult):,}; features the model sees {len(FEATURES)} "
      f"({len(CATEGORICAL)} categorical): {', '.join(FEATURES)}")
print(f"fit {len(i_fit):,}   validation {len(i_val):,}   test {len(i_test):,}")
print(f"share of women: {female_all.mean():.4f}")
print(f"share above 50K: all {y_all.mean():.4f}   men {y_all[female_all == 0].mean():.4f}"
      f"   women {y_all[female_all == 1].mean():.4f}")

## 3 · The model the bank ships

A histogram-based gradient-boosted ensemble, the model a practitioner fits on
a table of mixed columns. It reads the categorical codes as categories, not as
numbers, through `categorical_features`. The product rule is a threshold on its
probability: at 0.5 or above, the applicant is pre-approved instantly; below,
they go to the manual check.

> **Specification · the model the bank ships**
>
> **input** · `X_fit`, `y_fit`, and the validation and test matrices
>
> **output** · a fitted boosted ensemble, its probabilities on the validation and test rows, and its test accuracy and AUC beside the accuracy of refusing everyone
>
> **constraint** · the six categorical columns are declared as categorical, so a code is never read as a magnitude; the threshold is a named constant, `THRESHOLD = 0.5`, because every audit number below depends on it
>
> **check** · the accuracy of refusing everyone is one minus the share above 50K in the test rows; the model must beat it clearly, and its AUC must lie well above 0.5
>
> ---
>
> **novice prompt** · “train a gradient boosting classifier on the adult data”
>
> **pro prompt** · “Fit `HistGradientBoostingClassifier(categorical_features=CAT_MASK, random_state=RANDOM_STATE)` on `X_fit, y_fit`. Store the positive-class probabilities as `p_val` and `p_test`. Set `THRESHOLD = 0.5`. Print the test accuracy at the threshold, the test AUC, the accuracy of refusing everyone (one minus the test share above 50K), the share of test applicants pre-approved, and the number of boosting iterations early stopping kept.”
>
> **why the pro prompt works better** · 'Train a gradient boosting classifier on the adult data' gets a model, and asked this way an assistant typically uses `GradientBoostingClassifier` on the integer codes, so the occupation coded 7 is treated as larger than the one coded 3, and it reports accuracy alone, with no anchor to say whether 0.87 is good when three quarters of the rows are one class. It leaves the threshold inside `.predict()`, where no later cell can name it. The pro's `categorical_features=CAT_MASK`, the AUC beside the refuse-everyone anchor and the named `THRESHOLD` make the model's quality and its decision rule two stated quantities.
>
> ---
>
> **try** · drop `categorical_features=CAT_MASK`. The codes are then read as numbers; accuracy and AUC barely move, because trees can carve any ordering into pieces — but the importances and Shapley values below are now about an ordering nobody chose.

In [ ]:
model = HistGradientBoostingClassifier(categorical_features=CAT_MASK,
                                       random_state=RANDOM_STATE).fit(X_fit, y_fit)
p_val  = model.predict_proba(X_val)[:, 1]
p_test = model.predict_proba(X_test)[:, 1]

THRESHOLD = 0.5            # the product rule: pre-approve at or above it
approve = p_test >= THRESHOLD

auc_test = roc_auc_score(y_test, p_test)
acc_test = (approve == y_test).mean()
print(f"boosting iterations kept by early stopping: {model.n_iter_}")
print(f"test accuracy at {THRESHOLD}: {acc_test:.4f}")
print(f"refuse everyone:        {1 - y_test.mean():.4f}")
print(f"test AUC:               {auc_test:.4f}")
print(f"share pre-approved:     {approve.mean():.4f}")

## 4 · The importance the library offers, and why it misleads

A random forest, the ensemble whose `feature_importances_` most people have
met, fitted to the same rows. Its importance is the **impurity** decrease
summed over every split on a column — earned on the training rows the trees
were grown on. Beside it, **permutation importance**: shuffle one column,
re-score, report the fall in AUC. Computed twice, on the training rows and on
the held-out test rows.

> **Specification · impurity against permutation**
>
> **input** · `X_fit`, `y_fit`, `X_test`, `y_test`
>
> **output** · a forest of 200 trees, and for each feature its impurity importance and its permutation importance in AUC on the training rows and on the test rows, with the rank under each
>
> **constraint** · the permutation importances are falls in AUC with five repeats each; the forest's training and test AUC are printed beside them, because the gap between those two is what the training-row importances measure
>
> **check** · impurity importances sum to 1; a fully grown forest scores an AUC very close to 1 on its own training rows, so any importance computed there is partly about memorising them
>
> ---
>
> **novice prompt** · “show the feature importances of a random forest”
>
> **pro prompt** · “Fit `RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)` on `X_fit, y_fit`. Put its `feature_importances_` in a Series over `FEATURES`. Run `permutation_importance` with `scoring="roc_auc"`, `n_repeats=5` and `RANDOM_STATE` twice: on `X_fit, y_fit` and on `X_test, y_test`. Print one table with the three columns and a rank for each, sorted by impurity, and print the forest's training and test AUC.”
>
> **why the pro prompt works better** · 'Show the feature importances of a random forest' gets `feature_importances_` plotted as a bar chart, which is exactly what the request says. Asked this way, an assistant does not compute a second measure, and does not say that the bars are impurity decreases summed over the training rows, so nothing on the chart can show that a column with thousands of distinct values earns importance by being split on, not by predicting. The pro's three columns — impurity, permutation on the training rows, permutation on the test rows — with the forest's two AUCs printed beside them put the cause of any disagreement on the table.
>
> ---
>
> **try** · refit the forest with `min_samples_leaf=50`. Its training AUC falls towards its test AUC, and `fnlwgt`'s impurity importance falls with it — the importance was paid for by leaves too small to generalise.

In [ ]:
forest = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                n_jobs=-1).fit(X_fit, y_fit)
# predict on one thread: the trees' probabilities are then summed in a fixed
# order, and the importances below reproduce to the last digit
forest.set_params(n_jobs=1)
print(f"forest AUC, training rows {roc_auc_score(y_fit, forest.predict_proba(X_fit)[:, 1]):.4f}"
      f"   test rows {roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1]):.4f}")

impurity   = pd.Series(forest.feature_importances_, index=FEATURES)
perm_train = permutation_importance(forest, X_fit, y_fit, scoring="roc_auc",
                                    n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
perm_test  = permutation_importance(forest, X_test, y_test, scoring="roc_auc",
                                    n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)

forest_table = pd.DataFrame({
    "impurity": impurity,
    "perm, training": pd.Series(perm_train.importances_mean, index=FEATURES),
    "perm, test": pd.Series(perm_test.importances_mean, index=FEATURES),
})
for col in list(forest_table.columns):
    forest_table["rank: " + col] = forest_table[col].rank(ascending=False).astype(int)
forest_table = forest_table.sort_values("impurity", ascending=False)
assert abs(impurity.sum() - 1) < 1e-9
print(forest_table.round(4).to_string())
print(f"\nfnlwgt, the census sampling weight: impurity {impurity['fnlwgt']:.4f}, "
      f"permutation on test {forest_table.loc['fnlwgt', 'perm, test']:.4f}")
distinct = {c: len(np.unique(X_fit[:, FEATURES.index(c)])) for c in ("fnlwgt", "age", "education-num")}
print("distinct values on the fit rows:", {c: f"{n:,}" for c, n in distinct.items()})

## 5 · Permutation importance of the shipped model

Permutation importance needs nothing from the model but its predictions, so it
works for the boosted ensemble as for anything else. Written out by hand here,
so that every step is visible: shuffle one column, re-score, report the fall in
AUC, ten times.

> **Specification · permutation importance of the shipped model**
>
> **input** · the fitted `model`, `X_test`, `y_test`
>
> **output** · for every feature the mean and spread of the fall in test AUC when it is shuffled, over ten repeats
>
> **constraint** · each column is shuffled with a permutation of the rows, so it keeps its own values and loses only its link to the rest of the row
>
> **check** · a feature the model never splits on scores zero exactly; the spread over repeats is small beside the larger falls
>
> ---
>
> **novice prompt** · “compute permutation importance for the boosting model”
>
> **pro prompt** · “Write `perm_drop(model, X, y, cols, n_repeats=10, seed=RANDOM_STATE)` returning the mean and standard deviation of the fall in `roc_auc_score` when the columns in `cols` are replaced by a row permutation of themselves, drawn from `np.random.default_rng(seed)`. Apply it to every feature of `FEATURES` on the test rows and print them sorted.”
>
> **why the pro prompt works better** · 'Compute permutation importance for the boosting model' gets one call to `permutation_importance`, which is right as far as it goes. Asked this way, an assistant typically scores it with accuracy, the default, which barely moves on a table where three quarters of the rows are one class. The pro's AUC and its spread over ten repeats are what make the falls comparable and show which differences are noise.
>
> ---
>
> **try** · add a column of random numbers to `X_test` and to the model's training data, refit, and shuffle it like the others. Its fall is the noise floor: a real column that scores no better than it is not shown to matter.

In [ ]:
def perm_drop(model, X, y, cols, n_repeats=10, seed=RANDOM_STATE):
    """Fall in AUC when the columns in `cols` are shuffled, as (mean, sd)."""
    rng = np.random.default_rng(seed)
    j = [FEATURES.index(c) for c in cols]
    base = roc_auc_score(y, model.predict_proba(X)[:, 1])
    drops = []
    for _ in range(n_repeats):
        Xp = X.copy()
        Xp[:, j] = X[rng.permutation(len(X))][:, j]
        drops.append(base - roc_auc_score(y, model.predict_proba(Xp)[:, 1]))
    return float(np.mean(drops)), float(np.std(drops))

single = pd.DataFrame([perm_drop(model, X_test, y_test, [c]) for c in FEATURES],
                      index=FEATURES, columns=["AUC fall", "sd"])
single = single.sort_values("AUC fall", ascending=False)
print(single.round(4).to_string())

## 6 · Partial dependence and ICE

**Individual conditional expectation** (ICE): take one applicant, set their
age to each value on a grid, keep every other answer, and read the model's
probability — one curve per applicant. The **partial dependence** (PDP) is the
average of those curves. Computed by hand here on a fixed sample of 1,000 test
applicants; `sklearn.inspection.partial_dependence` does the same.

> **Specification · ICE and partial dependence on a toy model**
>
> **input** · a model written by hand, `toy(age, married)`, and four applicants
>
> **output** · each applicant's ICE curve at ages 20, 40 and 60, their own prediction at their own age, and the partial dependence, the mean of the four curves
>
> **constraint** · a model simple enough to check every number by hand: 0.10 for the unmarried at any age; for the married, 0.10 at 20 rising by 0.10 every ten years
>
> **check** · the married curves rise by 0.40 from 20 to 60, the others stay flat, and the partial dependence rises by 0.20, half of it, which no applicant does
>
> ---
>
> **novice prompt** · “explain partial dependence with a simple example”
>
> **pro prompt** · “Write `toy(age, married)` returning 0.10 + 0.40 · married · (age − 20) / 40. Make four applicants as (age, married): (30, 1), (55, 1), (35, 0), (60, 0). For each, print its own prediction and its ICE curve on `TOY_AGES = [20, 40, 60]` (age replaced, married kept); then print the partial dependence, the mean of the four curves, and assert it equals the mean of the married and the unmarried curves.”
>
> **why the pro prompt works better** · 'Explain partial dependence with a simple example' gets a paragraph, or a library plot on a real dataset where no number can be checked by hand. Four applicants and a model written in one line make every value checkable on paper, and show the point before the real data does: the average curve can rise at a rate no single applicant has.
>
> ---
>
> **try** · make the unmarried fall with age instead, `- 0.40 * (age - 20) / 40` for them, starting from 0.50. The married still rise and the unmarried fall by the same amount, and the partial dependence is flat: an average that says age does nothing.

In [ ]:
def toy(age, married):
    """A model written by hand: flat for the unmarried, rising with age for the married."""
    return 0.10 + 0.40 * married * (age - 20) / 40

TOY_AGES = [20, 40, 60]
TOY = {"A": (30, 1), "B": (55, 1), "C": (35, 0), "D": (60, 0)}

toy_ice = {k: [toy(a, m) for a in TOY_AGES] for k, (age, m) in TOY.items()}
toy_pdp = np.mean(list(toy_ice.values()), axis=0)
assert np.allclose(toy_pdp, (np.mean([toy_ice["A"], toy_ice["B"]], 0)
                             + np.mean([toy_ice["C"], toy_ice["D"]], 0)) / 2)

print("applicant  age  married  own prediction   ICE at 20    40    60")
for k, (age, m) in TOY.items():
    c = toy_ice[k]
    print(f"    {k}      {age}     {m}         {toy(age, m):.2f}          "
          f"{c[0]:.2f}   {c[1]:.2f}  {c[2]:.2f}")
print(f"partial dependence                          "
      f"{toy_pdp[0]:.2f}   {toy_pdp[1]:.2f}  {toy_pdp[2]:.2f}")
print(f"\nrise from 20 to 60: married {toy_ice['A'][2] - toy_ice['A'][0]:.2f}, "
      f"unmarried {toy_ice['C'][2] - toy_ice['C'][0]:.2f}, "
      f"partial dependence {toy_pdp[2] - toy_pdp[0]:.2f}")

> **Specification · partial dependence and ICE**
>
> **input** · the fitted `model` and 1,000 test applicants drawn with `RANDOM_STATE`
>
> **output** · an ICE matrix, one row per applicant and one column per age from 18 to 80 in steps of two, its column mean (the PDP), and the mean curves of the married and of everyone else
>
> **constraint** · probabilities, not log-odds, so the curves read in the units of the product rule; the same 1,000 applicants for every curve
>
> **check** · the ICE matrix is 1,000 × 32; the PDP is the weighted average of the two group curves, with the share married as the weight
>
> ---
>
> **novice prompt** · “plot the partial dependence of income on age”
>
> **pro prompt** · “Draw 1,000 test rows with `np.random.default_rng(RANDOM_STATE).choice(..., replace=False)` as `X_s`. Write `ice(model, X, col, grid)` returning the matrix of `predict_proba(...)[:, 1]` with column `col` set to each grid value in turn. Compute it for age on `AGES = np.arange(18, 81, 2)`, take the column mean as the PDP, and split the curves by whether `marital-status` is `Married-civ-spouse`. Print the PDP and the two group means at ages 20 to 80 in steps of ten, and the share married; assert the PDP equals the weighted average of the two group means.”
>
> **why the pro prompt works better** · 'Plot the partial dependence of income on age' gets `PartialDependenceDisplay.from_estimator`, and asked this way an assistant typically draws the average only. For a boosted model the library's fast path averages the log-odds, not the probability, so the curve is in units nobody decides with, and the one line hides whether all applicants move together or two groups move differently. The pro's ICE matrix in probabilities, split by marital status, and the assert that the average is the weighted mean of the groups make the average and what it averages both visible.
>
> ---
>
> **try** · split the ICE curves by `hours-per-week` above 40 instead of by marital status. The two group curves still separate, but by less than the married and the rest: hours interact with age too, more weakly than marriage does.

In [ ]:
X_s = X_test[np.random.default_rng(RANDOM_STATE).choice(len(X_test), 1000, replace=False)]

def ice(model, X, col, grid):
    """One row per applicant, one column per grid value: the model's probability
    with `col` set to that value and every other answer kept."""
    j = FEATURES.index(col)
    out = np.empty((len(X), len(grid)))
    for k, g in enumerate(grid):
        Xg = X.copy()
        Xg[:, j] = g
        out[:, k] = model.predict_proba(Xg)[:, 1]
    return out

AGES = np.arange(18, 81, 2)
ice_age = ice(model, X_s, "age", AGES)
pdp_age = ice_age.mean(0)
married = X_s[:, FEATURES.index("marital-status")] == LEVELS["marital-status"].index("Married-civ-spouse")
age_married, age_other = ice_age[married].mean(0), ice_age[~married].mean(0)

assert ice_age.shape == (1000, len(AGES))
assert np.allclose(pdp_age, married.mean() * age_married + (1 - married.mean()) * age_other)
print(f"share married in the sample: {married.mean():.3f}")
print(" age    PDP   married   everyone else")
for a in range(20, 81, 10):
    k = int(np.where(AGES == a)[0][0])
    print(f"{a:4d}  {pdp_age[k]:.3f}    {age_married[k]:.3f}        {age_other[k]:.3f}")

## 7 · Shapley values, exactly, for one applicant

The value of a coalition $S$ of features is the model's mean prediction when
the features in $S$ take the applicant's values and the rest take the values
of a background applicant, averaged over a background sample:
$v(S) = \frac{1}{|B|}\sum_{\mathbf{b}\in B} f(\mathbf{x}_S, \mathbf{b}_{\bar S})$.
The Shapley value of feature $i$ is its marginal contribution averaged over
every coalition, with the weights that make it an average over orderings:

$$\phi_i = \sum_{S \subseteq N\setminus\{i\}} \frac{|S|!\,(M-|S|-1)!}{M!}\,\bigl(v(S\cup\{i\}) - v(S)\bigr).$$

With $M = 12$ features there are $2^{12} = 4{,}096$ coalitions: few enough to
enumerate every one. **Efficiency**: the $\phi_i$ add up to
$v(N) - v(\varnothing)$, the prediction minus the background mean.

> **Specification · Shapley values, exactly**
>
> **input** · the model, 200 background rows drawn from the fit set, and one test applicant
>
> **output** · the twelve Shapley values of that applicant's probability, the baseline v(∅) and the prediction v(N)
>
> **constraint** · every coalition is enumerated — no sampling — and v(S) is a mean over the same 200 background rows for every S; the applicant is chosen by a rule, not by eye: among the qualified applicants (above 50K) the model refused, the one at the median probability
>
> **check** · two checks whose answers are known before running: on a linear function w·z the Shapley value of feature i is w_i (x_i − mean of the background's column i); and on the model, the values add up to the prediction minus the baseline, to floating-point precision
>
> ---
>
> **novice prompt** · “compute SHAP values for one prediction without the shap library”
>
> **pro prompt** · “Write `shapley_exact(predict, x, background)`: enumerate all 2^M coalitions as boolean masks with `itertools.product`, build one array of shape (2^M, |B|, M) holding x where the mask is true and the background row elsewhere, call `predict` once on it reshaped to rows, average over the background to get v for every coalition, then sum the weighted marginal contributions `w(|S|) * (v(S ∪ {i}) - v(S))` over the coalitions without i. Return the values, v(∅) and v(N). Check it first on `lambda Z: Z @ w` with w = 1..M on NaN-free copies against w * (x - background.mean(0)). Background: 200 rows of `X_fit` drawn with `RANDOM_STATE`. Applicant: among test rows with y = 1 and probability below `THRESHOLD`, sorted by probability, the one in the middle. Print the applicant's answers decoded, the baseline, the prediction and the values sorted by size, and assert they add up.”
>
> **why the pro prompt works better** · 'Compute SHAP values for one prediction without the shap library' is a request for an approximation. Asked this way, an assistant typically writes a permutation-sampling or KernelSHAP-style estimator with a few hundred random orderings and a background it draws inside the function, so the values change from run to run and their sum misses the prediction by an amount nobody checks. It picks the applicant by index. The pro's full enumeration, one fixed background, the linear check with a known answer and the assert on efficiency make the values exact and checked, and the stated rule for the applicant makes the explanation one the bank could be asked for.
>
> ---
>
> **try** · use the 200 background rows of the test set's refused applicants instead of the fit set. The baseline v(∅) drops, every value is now measured against a different 'typical applicant', and they still add up to the same prediction minus the new baseline.

In [ ]:
background = X_fit[np.random.default_rng(RANDOM_STATE).choice(len(X_fit), 200, replace=False)]
M = len(FEATURES)
WEIGHT = np.array([math.factorial(s) * math.factorial(M - s - 1) / math.factorial(M)
                   for s in range(M)])

def shapley_exact(predict, x, background):
    """Exact interventional Shapley values of predict at x, against a background."""
    masks = np.array(list(itertools.product([False, True], repeat=M)))   # 2^M x M
    rows = np.where(masks[:, None, :], x[None, None, :], background[None, :, :])
    v = predict(rows.reshape(-1, M)).reshape(len(masks), -1).mean(1)      # v(S), every S
    # row k of `masks` is the binary number k, first feature the highest bit
    bit = 1 << np.arange(M)[::-1]
    index, size = masks @ bit, masks.sum(1)
    phi = np.zeros(M)
    for i in range(M):
        without_i = ~masks[:, i]
        phi[i] = (WEIGHT[size[without_i]]
                  * (v[index[without_i] + bit[i]] - v[without_i])).sum()
    return phi, v[0], v[-1]

# A check with a known answer: for a linear function the value is w_i (x_i - mean b_i).
w = np.arange(1, M + 1, dtype=float)
x0, b0 = np.nan_to_num(X_test[0]), np.nan_to_num(background)
phi_lin, _, _ = shapley_exact(lambda Z: Z @ w, x0, b0)
assert np.allclose(phi_lin, w * (x0 - b0.mean(0)))

f = lambda Z: model.predict_proba(Z)[:, 1]
# the rule: among the qualified applicants the model refused, the one at the
# median probability -- a typical refusal, not an extreme one
missed = np.where((y_test == 1) & (p_test < THRESHOLD))[0]
missed = missed[np.argsort(p_test[missed], kind="stable")]
who = int(missed[len(missed) // 2])
x = X_test[who]
phi, baseline, fx = shapley_exact(f, x, background)

assert abs(fx - p_test[who]) < 1e-9
assert abs(baseline + phi.sum() - fx) < 1e-9          # efficiency
print(f"coalitions {2 ** M:,}; predictions for one applicant {2 ** M * len(background):,}")
print(f"qualified applicants refused: {len(missed)}; the median one is test row {who}, "
      f"{'a woman' if female_test[who] else 'a man'}")
print(f"probability {fx:.4f} against the threshold {THRESHOLD}")
answers = {c: ("missing" if np.isnan(v) else LEVELS[c][int(v)] if c in LEVELS else int(v))
           for c, v in zip(FEATURES, x)}
print("answers:", answers)
print(f"\nbaseline v(empty set), the background's mean probability: {baseline:.4f}")
shap_one = pd.Series(phi, index=FEATURES).sort_values(key=np.abs, ascending=False)
print(shap_one.round(4).to_string())
print(f"\nsum of the values {phi.sum():.4f}; baseline + sum = {baseline + phi.sum():.4f} "
      f"= the prediction {fx:.4f}")
blank = np.isnan(X_fit[:, FEATURES.index("occupation")])
print(f"fit rows with occupation left blank: {blank.mean():.4f} of them; share above 50K "
      f"{y_fit[blank].mean():.4f} against {y_fit[~blank].mean():.4f} for the rest")

## 8 · Local values, averaged: the global picture again

One applicant's values say what moved *their* probability. Averaged in
absolute value over many applicants they give a global ranking, which can be
set beside the permutation ranking of section 5. Forty test applicants, the
same background.

> **Specification · local against global**
>
> **input** · `shapley_exact`, the model, the background, and 40 test applicants drawn with `RANDOM_STATE`
>
> **output** · the mean absolute Shapley value of each feature over the 40, beside the single-column permutation importance, with both ranks
>
> **constraint** · the same background and the same exact enumeration as for the one applicant, so the two sections differ only in who is explained
>
> **check** · every applicant's values add up to that applicant's prediction minus the same baseline; the baseline is identical for all 40 because it depends on the background alone
>
> ---
>
> **novice prompt** · “compute global feature importance from SHAP values”
>
> **pro prompt** · “Draw 40 test indices with `np.random.default_rng(RANDOM_STATE).choice(..., replace=False)`, run `shapley_exact` with `f` and `background` on each, assert that every row of values adds to its prediction minus the baseline, and print a table over `FEATURES` of the mean absolute value with its rank beside the `AUC fall` of `single` with its rank, sorted by the first.”
>
> **why the pro prompt works better** · 'Compute global feature importance from SHAP values' invites the library summary plot, and without the library an assistant typically re-implements a sampled estimator with a fresh background inside each call, so the forty explanations are measured against forty different baselines and their average mixes them. It reports the ranking alone, with nothing to compare it to. The pro's one background, the efficiency assert per applicant and the permutation ranks printed beside make agreement or disagreement between the local and the global picture something the table shows.
>
> ---
>
> **try** · average the signed values instead of the absolute ones. Most of them fall towards zero — positive for some applicants, negative for others — which is why the global summary of a local explanation is a mean absolute value.

In [ ]:
some = np.random.default_rng(RANDOM_STATE).choice(len(X_test), 40, replace=False)
PHI = []
for k in some:
    phi_k, base_k, f_k = shapley_exact(f, X_test[k], background)
    assert abs(base_k - baseline) < 1e-12 and abs(base_k + phi_k.sum() - f_k) < 1e-9
    PHI.append(phi_k)
PHI = np.array(PHI)

local_global = pd.DataFrame({
    "mean |Shapley|": np.abs(PHI).mean(0),
    "AUC fall": single["AUC fall"].reindex(FEATURES).to_numpy(),
}, index=FEATURES)
local_global["rank, Shapley"] = local_global["mean |Shapley|"].rank(ascending=False).astype(int)
local_global["rank, permutation"] = local_global["AUC fall"].rank(ascending=False).astype(int)
local_global = local_global.sort_values("mean |Shapley|", ascending=False)
print(local_global.round(4).to_string())

## 9 · The audit, by slice

From here the question is not *what does the model rely on* but *for whom does
it fail*. The first instrument is the familiar one — accuracy, TPR, FPR,
PPV, AUC — read per slice of the applicants instead of once over all of them: by sex, by
age band, by race. Every rate comes with its count, because a rate on fifteen
people is not a rate on fifteen hundred.

> **Specification · metrics per slice**
>
> **input** · `p_test`, `y_test`, `female_test`, `race_test` and the test ages
>
> **output** · one row per slice — sex, four age bands, five race groups — with the count, the number above 50K, the base rate, the share pre-approved, accuracy, TPR, FPR, PPV and AUC
>
> **constraint** · every rate is computed at `THRESHOLD` on the test rows, and the count of positives is printed beside it, so a rate on a handful of people cannot pass for a finding
>
> **check** · the slices of one attribute partition the test rows, so their counts add to 9,769 for sex, for age and for race
>
> ---
>
> **novice prompt** · “evaluate the model separately for each group”
>
> **pro prompt** · “Write `rates(y, p, t=THRESHOLD)` returning a dict with n, positives, base rate, selected (share with p ≥ t), accuracy, TPR, FPR, PPV and AUC. Apply it to the test rows split by sex, by age band (under 25, 25–44, 45–64, 65 and over) and by `race_test`; assert that each attribute's counts sum to the test size; print one table, rates to four decimals.”
>
> **why the pro prompt works better** · 'Evaluate the model separately for each group' gets a groupby and an accuracy per group, and asked this way an assistant typically stops there, or adds a classification report at the default threshold. Accuracy per group is the number that misleads most here: a group with few people above 50K is easy to be right about by refusing. Counts are usually left out, so a rate on fifteen positives looks as solid as one on two thousand. The pro's TPR, FPR and PPV beside the base rate, at the named threshold, with the counts printed, make every row readable for what it is.
>
> ---
>
> **try** · add a slice for applicants whose `native-country` is missing. Its count is small and its rates swing; print it anyway, because a slice the model knows least about is the one an audit must not skip.

In [ ]:
def rates(y, p, t=THRESHOLD):
    a = p >= t
    return {"n": len(y), "positives": int(y.sum()), "base rate": y.mean(),
            "selected": a.mean(), "accuracy": (a == y).mean(),
            "TPR": a[y == 1].mean(), "FPR": a[y == 0].mean(),
            "PPV": y[a].mean() if a.any() else np.nan,
            "AUC": roc_auc_score(y, p)}

age_test = X_test[:, FEATURES.index("age")]
SLICES = {"men": female_test == 0, "women": female_test == 1,
          "age under 25": age_test < 25, "age 25-44": (age_test >= 25) & (age_test < 45),
          "age 45-64": (age_test >= 45) & (age_test < 65), "age 65+": age_test >= 65}
SLICES.update({f"race {r}": race_test == r for r in sorted(set(race_test))})

for group in (["men", "women"], [s for s in SLICES if s.startswith("age")],
              [s for s in SLICES if s.startswith("race")]):
    assert sum(SLICES[s].sum() for s in group) == len(y_test)

slices = pd.DataFrame({s: rates(y_test[m], p_test[m]) for s, m in SLICES.items()}).T
slices[["n", "positives"]] = slices[["n", "positives"]].astype(int)
print(slices.round(4).to_string())

## 10 · Three fairness criteria, measured

With $\hat Y$ the decision, $Y$ the truth and $A$ the group:

- **demographic parity**: $P(\hat Y = 1 \mid A = a)$ equal across groups — the
  same share pre-approved;
- **equal opportunity**: $P(\hat Y = 1 \mid Y = 1, A = a)$ equal — the same
  TPR, so a qualified applicant's chance does not depend on the group;
- **calibration within groups**: $P(Y = 1 \mid s, A = a) = s$ for every score
  $s$ — a probability means the same thing in every group. At a threshold, its
  footprint is equal PPV.

When the base rates differ, these cannot all hold. For one group with base
rate $p$, count its true positives two ways — as a share of the people above
50K, and as a share of the people selected:
$$\text{TPR} = \frac{\text{selected} \times \text{PPV}}{p}.$$
Hold any two of selection rate, PPV and TPR equal across the groups, and the
identity fixes the third by the ratio of the base rates.

> **Specification · three fairness criteria**
>
> **input** · the `rates` of men and of women at `THRESHOLD`
>
> **output** · the selection rates and their ratio, the TPRs and their gap, the PPVs and their gap, the FPRs, the identity checked for each group, and the selection ratio the identity would force if PPV and TPR were equal
>
> **constraint** · the identity is checked as an assert, from the group's own selection rate, PPV and base rate — it is algebra, so it must hold to floating-point precision
>
> **check** · with PPV and TPR equal, the identity makes the selection ratio women/men equal to the ratio of their base rates, which can be computed from section 2's numbers before running
>
> ---
>
> **novice prompt** · “check if the model is fair between men and women”
>
> **pro prompt** · “From `rates` on the test rows of each sex, print the base rate, selection rate, TPR, FPR and PPV of each group; then the selection ratio women/men, the TPR gap and the PPV gap. For each group assert that TPR equals selected × PPV / base rate to 1e-12. Print the ratio of the base rates women/men beside the measured selection ratio.”
>
> **why the pro prompt works better** · 'Check if the model is fair between men and women' has no single answer, and asked this way an assistant typically picks one criterion, usually demographic parity with the four-fifths rule, and returns a verdict. The verdict depends entirely on the criterion chosen, and with the base rates this far apart the three criteria disagree, so the request hides the finding. The pro's three criteria side by side and the identity, asserted per group, show which ones hold, which do not, and why they cannot all hold together.
>
> ---
>
> **try** · raise `THRESHOLD` to 0.7 and rerun this cell. The selection ratio and the TPR gap both change; the identity still holds for each group, because it is algebra, not a property of the model.

In [ ]:
G = {"men": rates(y_test[female_test == 0], p_test[female_test == 0]),
     "women": rates(y_test[female_test == 1], p_test[female_test == 1])}
for g, r in G.items():
    # the true positives, counted as a share of the qualified and of the selected
    assert abs(r["selected"] * r["PPV"] / r["base rate"] - r["TPR"]) < 1e-12
    print(f"{g:6s} base rate {r['base rate']:.4f}  selected {r['selected']:.4f}"
          f"  TPR {r['TPR']:.4f}  FPR {r['FPR']:.4f}  PPV {r['PPV']:.4f}")

G_m, G_w = G["men"], G["women"]
base_ratio = G_w["base rate"] / G_m["base rate"]
print(f"\ndemographic parity: selection ratio women/men {G_w['selected'] / G_m['selected']:.4f}")
print(f"equal opportunity:  TPR gap men - women {G_m['TPR'] - G_w['TPR']:.4f}")
print(f"predictive parity:  PPV gap men - women {G_m['PPV'] - G_w['PPV']:.4f}")
print(f"base-rate ratio women/men {base_ratio:.4f}: the selection ratio that equal PPV "
      f"and equal TPR would force")

## 11 · Calibration within groups

Calibration, read per group: bin the probabilities, and in each bin compare
the mean probability with the share that actually earned above 50K.

> **Specification · calibration within groups**
>
> **input** · `p_test`, `y_test`, `female_test`
>
> **output** · for each sex and each of ten equal-width probability bins, the count, the mean probability and the observed share above 50K; and per group the mean probability beside the base rate and the count-weighted mean gap
>
> **constraint** · the bins are the same for both groups, and a bin is reported with its count, because the top bins hold few women and their observed share is noisy
>
> **check** · the mean probability of a group and its base rate agree to about a point if the model is calibrated in that group; the counts of each group's bins add to the group's size
>
> ---
>
> **novice prompt** · “plot calibration curves for men and women”
>
> **pro prompt** · “For each sex, assign the test probabilities to ten bins with edges `np.linspace(0, 1, 11)`, and print per bin the count, the mean probability and the observed share. Assert the counts sum to the group's size. Print per group the mean probability, the base rate and the expected calibration error, the count-weighted mean of the absolute gap between mean probability and observed share.”
>
> **why the pro prompt works better** · 'Plot calibration curves for men and women' gets `CalibrationDisplay` twice, which draws the curves and hides the counts. Asked this way, an assistant typically uses quantile bins per group, so the two curves are evaluated at different probabilities and cannot be compared bin by bin, and a bin of a couple of dozen people is drawn with the same weight as one of two thousand. The pro's shared equal-width bins, the counts printed and the count-weighted error per group make 'calibrated within each group' a number and say how much each bin can be trusted.
>
> ---
>
> **try** · use five bins instead of ten. The women's calibration error falls, because each bin now holds more of them — much of the gap was noise in small counts, not a fault of the model.

In [ ]:
EDGES = np.linspace(0, 1, 11)
calib = {}
for g, mask in (("men", female_test == 0), ("women", female_test == 1)):
    pg, yg = p_test[mask], y_test[mask]
    b = np.clip(np.digitize(pg, EDGES[1:-1]), 0, 9)
    rows = [(int((b == k).sum()), pg[b == k].mean(), yg[b == k].mean())
            for k in range(10) if (b == k).any()]
    assert sum(r[0] for r in rows) == mask.sum()
    ece = sum(n * abs(pm - ym) for n, pm, ym in rows) / mask.sum()
    calib[g] = {"bins": rows, "mean p": pg.mean(), "base rate": yg.mean(), "ECE": ece}
    print(f"{g}: mean probability {pg.mean():.4f}, base rate {yg.mean():.4f}, "
          f"calibration error {ece:.4f}")
    for n, pm, ym in rows:
        print(f"   n {n:5d}   mean probability {pm:.3f}   observed {ym:.3f}")

## 12 · Dropping the column does not drop the information

The model was never given `sex`. If the twelve answers it does see predict
sex well, the model can reach it anyway — and whatever it learned about the
target through those answers is not removed by removing the column. Measure
it: fit a classifier whose target is sex.

> **Specification · proxies**
>
> **input** · `X_fit`, `female_fit`, `X_test`, `female_test`
>
> **output** · the test AUC of predicting sex from the twelve features, then without `relationship`, then without `relationship` and `marital-status`
>
> **constraint** · the same model class and settings as the bank's model, so the question is what that model could extract; fitted on the fit rows and scored on the test rows
>
> **check** · `relationship` has the levels Husband and Wife, which name the sex of every married applicant; with it the AUC must be high, and removing it must lower the AUC without taking it to 0.5
>
> ---
>
> **novice prompt** · “check whether other features are correlated with sex”
>
> **pro prompt** · “Write `proxy_auc(drop)` that fits `HistGradientBoostingClassifier(categorical_features=CAT_MASK[keep], random_state=RANDOM_STATE)` on the fit rows of every feature not in `drop`, with `female_fit` as the target, and returns `roc_auc_score` on the test rows. Print it for no column dropped, for `relationship` dropped and for both `relationship` and `marital-status` dropped.”
>
> **why the pro prompt works better** · 'Check whether other features are correlated with sex' gets a correlation matrix or Cramér's V for each column against sex. Asked this way, an assistant measures one column at a time, so a combination — hours worked with occupation with marital status — that identifies sex together while each says little alone is never seen, and a correlation is not what the bank needs to know. The pro's classifier, of the same class as the bank's model, scored by AUC with the strongest proxies removed one after the other, measures what the model could actually extract.
>
> ---
>
> **try** · also drop `occupation`. The AUC falls again and is still well above 0.5: there is no single column whose removal makes the attribute unreachable, which is why the audit measures outcomes rather than inputs.

In [ ]:
def proxy_auc(drop):
    keep = [j for j, c in enumerate(FEATURES) if c not in drop]
    clf = HistGradientBoostingClassifier(categorical_features=CAT_MASK[keep],
                                         random_state=RANDOM_STATE)
    clf.fit(X_fit[:, keep], female_fit)
    return roc_auc_score(female_test, clf.predict_proba(X_test[:, keep])[:, 1])

proxy = {"all twelve features": proxy_auc([]),
         "without relationship": proxy_auc(["relationship"]),
         "without relationship and marital-status": proxy_auc(["relationship", "marital-status"])}
for k, v in proxy.items():
    print(f"AUC for predicting sex, {k:40s} {v:.4f}")

## 13 · Moving the threshold for one group

The product rule is one threshold for everyone. Two alternatives keep the
men's threshold at 0.5 and move the women's, chosen on the **validation** rows
and read on the test rows:

- **equal opportunity**: the highest women's threshold at which their
  validation TPR reaches the men's;
- **demographic parity**: the highest women's threshold at which their
  validation selection rate reaches the men's.

Both use the applicant's sex at the moment of the decision. Whether that is
allowed is a question for the bank's legal and policy owners, not for this
notebook; what the notebook can do is say what each rule costs and buys.

> **Specification · a threshold per group**
>
> **input** · `p_val`, `y_val`, `female_val` to choose; `p_test`, `y_test`, `female_test` to read
>
> **output** · for each of three policies — one threshold, equal opportunity, demographic parity — the women's threshold, and on the test rows the selection rate, TPR, FPR and PPV of each group, the overall accuracy, and the numbers of qualified and of unqualified women pre-approved
>
> **constraint** · the women's thresholds are chosen on the validation rows only; the men's stays at `THRESHOLD`; the test rows are read once, after every threshold is fixed
>
> **check** · the one-threshold row reproduces section 10 exactly; equal opportunity lowers the women's threshold, so it can only raise their TPR and their FPR; demographic parity lowers it further
>
> ---
>
> **novice prompt** · “adjust the thresholds so the model is fair to women”
>
> **pro prompt** · “Write `threshold_for(scores, rate)` returning the highest score s such that the share of `scores` at or above s is at least `rate`. On the validation rows compute the men's TPR and selection rate at `THRESHOLD`; set `t_eo` from the women's validation positives and the men's TPR, and `t_dp` from all the women's validation scores and the men's selection rate. For each policy decide on the test rows with `THRESHOLD` for men and the policy's threshold for women, and print one table with the thresholds, each group's selection rate, TPR, FPR and PPV and the overall accuracy, then the counts of qualified and unqualified women pre-approved.”
>
> **why the pro prompt works better** · 'Adjust the thresholds so the model is fair to women' leaves both the criterion and the data to choose on open. Asked this way, an assistant typically searches thresholds on the test rows until a gap closes, which makes the reported test numbers the ones that chose the threshold, and it reports the criterion it equalised without the ones it moved. The pro's two named criteria, thresholds fixed on the validation rows, and a table of every rate for both groups under all three policies make the choice a stated trade-off rather than a tuned number.
>
> ---
>
> **try** · choose `t_eo` on the test rows instead of the validation rows. The women's test TPR then matches the men's almost exactly — and the match is no longer a measurement, because the test rows chose it.

In [ ]:
def threshold_for(scores, rate):
    """The highest threshold that selects at least `rate` of `scores`."""
    s = np.sort(scores)[::-1]
    return float(s[int(np.ceil(rate * len(s))) - 1])

men_v, women_v = female_val == 0, female_val == 1
tpr_men_val = (p_val[men_v & (y_val == 1)] >= THRESHOLD).mean()
sel_men_val = (p_val[men_v] >= THRESHOLD).mean()
t_eo = threshold_for(p_val[women_v & (y_val == 1)], tpr_men_val)
t_dp = threshold_for(p_val[women_v], sel_men_val)

POLICIES = {"one threshold": THRESHOLD, "equal opportunity": t_eo,
            "demographic parity": t_dp}
women_t = female_test == 1
policy_rows, approved_women = {}, {}
for name, t_w in POLICIES.items():
    a = p_test >= np.where(women_t, t_w, THRESHOLD)
    row = {"women's threshold": t_w}
    for g, mask in (("men", ~women_t), ("women", women_t)):
        yy, aa = y_test[mask], a[mask]
        row.update({f"selected {g}": aa.mean(), f"TPR {g}": aa[yy == 1].mean(),
                    f"FPR {g}": aa[yy == 0].mean(), f"PPV {g}": yy[aa].mean()})
    row["accuracy"] = (a == y_test).mean()
    policy_rows[name] = row
    approved_women[name] = (int((a & women_t & (y_test == 1)).sum()),
                            int((a & women_t & (y_test == 0)).sum()))
policies = pd.DataFrame(policy_rows)

assert abs(policies.loc["TPR women", "one threshold"] - G["women"]["TPR"]) < 1e-12
assert t_dp <= t_eo <= THRESHOLD
print(f"validation, men at {THRESHOLD}: TPR {tpr_men_val:.4f}, selected {sel_men_val:.4f}")
print(policies.round(4).to_string())
print("\nwomen pre-approved on the test rows:  qualified   not qualified")
q0, u0 = approved_women["one threshold"]
for name, (q, u) in approved_women.items():
    print(f"  {name:20s} {q:21d} {u:15d}    change: {q - q0:+d} qualified, {u - u0:+d} not")

## What to take from this notebook

- The importance a library offers by default may be a property of the
  training fit. Permutation importance on held-out rows is a property of the
  model on data it has not seen.
- A partial dependence is an average over rows the model may never have
  seen; the ICE curves show what the average hides.
- Shapley values are exact here because twelve features allow it; they add up
  to the prediction minus the baseline, and they are measured against a
  background someone chose.
- An audit reads every rate per slice, with its count. Three reasonable
  fairness criteria disagree when base rates differ, and that is algebra;
  dropping a column does not remove what the others say about it; and moving
  a threshold for one group is a decision about people, made with the numbers
  of section 13 in front of whoever owns it.